# Exploratory Data Analysis

Goal: understand how tyre wear and fuel burn affect lap times before building the model.

## Load data

In [37]:
import pandas as pd
import matplotlib.pyplot as plt

laps = pd.read_parquet("../data/processed/laps_model.parquet")
print(laps.shape)

(78233, 37)


## Q1. How fast does each tyre compound degrade?

Measure the lap-time change per lap of tyre age inside each stint (one set of tyres), then compare SOFT, MEDIUM and HARD.

In [38]:
import numpy as np

stint_keys = ["Year", "Round", "Driver", "Stint"]
MIN_STINT_LAPS = 8

laps["StintLaps"] = laps.groupby(stint_keys)["LapNumber"].transform("size")
long_stints = laps[laps["StintLaps"] >= MIN_STINT_LAPS].copy()

print("stints total:", laps.groupby(stint_keys).ngroups)
print("stints >= 8 laps:", long_stints.groupby(stint_keys).ngroups)

stints total: 4227
stints >= 8 laps: 3646


In [39]:
def stint_slope(group):
    slope, intercept = np.polyfit(group["TyreLife"], group["LapTimeSec"], deg=1)
    return slope

slopes = (
    long_stints
    .groupby(stint_keys + ["Compound"])[["TyreLife", "LapTimeSec"]]
    .apply(stint_slope)
    .reset_index(name="SecPerLap")
)
slopes.head()

,Year,Round,Driver,Stint,Compound,SecPerLap
0,2022,1,ALB,1.0,SOFT,0.233491
1,2022,1,ALB,2.0,MEDIUM,0.072089
2,2022,1,ALO,1.0,SOFT,0.240267
3,2022,1,ALO,2.0,MEDIUM,0.066098
4,2022,1,ALO,3.0,HARD,0.135510


In [40]:
summary = (
    slopes.groupby("Compound")["SecPerLap"]
    .agg(["median", "count"])
    .loc[["SOFT", "MEDIUM", "HARD"]]
)
summary

,median,count
Compound,,
SOFT,0.030189,586
MEDIUM,0.006299,1554
HARD,-0.005012,1506


In [41]:
long_stints = long_stints.sort_values(stint_keys + ["LapNumber"])

long_stints["LapInStint"] = long_stints.groupby(stint_keys).cumcount() + 1

long_stints["StintStart"] = (
    long_stints.groupby(stint_keys)["LapTimeSec"]
    .transform(lambda times: times.iloc[:3].median())
)

long_stints["DeltaSec"] = long_stints["LapTimeSec"] - long_stints["StintStart"]

In [42]:
curve = (
    long_stints.groupby(["Compound", "LapInStint"])["DeltaSec"]
    .agg(["median", "count"])
    .reset_index()
)

curve = curve[(curve["count"] >= 50) & (curve["LapInStint"] <= 35)]
curve.head()

,Compound,LapInStint,median,count
0,HARD,1,0.0135,1506
1,HARD,2,0.0000,1506
2,HARD,3,0.0000,1506
3,HARD,4,-0.0650,1506
4,HARD,5,-0.0720,1506


In [43]:
import plotly.express as px

tyre_colors = {"SOFT": "#E10600", "MEDIUM": "#FFD700", "HARD": "#9E9E9E"}

fig = px.line(
    curve,
    x="LapInStint",
    y="median",
    color="Compound",
    color_discrete_map=tyre_colors,
    category_orders={"Compound": ["SOFT", "MEDIUM", "HARD"]},
    markers=True,
    labels={
        "LapInStint": "Laps on this set of tyres",
        "median": "Slower than start of stint (s)",
        "Compound": "Tyre",
    },
    title="How lap times change as tyres age (2022-2025)",
)
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.show()

In [44]:
fig.write_image("../images/degradation_curve.png", scale=2)

In [45]:
bins = [8, 15, 25, 100]
names = ["8-14 laps", "15-24 laps", "25+ laps"]

long_stints["StintLength"] = pd.cut(
    long_stints["StintLaps"], bins=bins, labels=names, right=False
)
long_stints.groupby("StintLength", observed=True)[stint_keys].nunique()

,Year,Round,Driver,Stint
StintLength,,,,
8-14 laps,4,24,31,6
15-24 laps,4,24,31,6
25+ laps,4,24,31,5


In [46]:
curve_len = (
    long_stints.groupby(["Compound", "StintLength", "LapInStint"], observed=True)["DeltaSec"]
    .agg(["median", "count"])
    .reset_index()
)
curve_len = curve_len[curve_len["count"] >= 20]

In [47]:
fig3 = px.line(
    curve_len,
    x="LapInStint",
    y="median",
    color="StintLength",
    facet_col="Compound",
    category_orders={"Compound": ["SOFT", "MEDIUM", "HARD"]},
    markers=True,
    labels={"LapInStint": "Laps on this set of tyres",
            "median": "Slower than start of stint (s)",
            "StintLength": "Stint length"},
    title="Tyre degradation by stint length",
)
fig3.add_hline(y=0, line_dash="dash", line_color="grey")
fig3.show()

In [48]:
fig3.write_image("../images/degradation_by_stint_length.png", scale=2, width=1400, height=500)

## Q2. How much does fuel burn speed the car up, and what is the true tyre wear?

In [49]:
data = laps.copy()

for c in ["SOFT", "MEDIUM", "HARD"]:
    is_c = (data["Compound"] == c).astype(float)
    data[f"Age_{c}"] = data["TyreLife"] * is_c
    data[f"Is_{c}"] = is_c

features = ["LapNumber", "Age_SOFT", "Age_MEDIUM", "Age_HARD", "Is_SOFT", "Is_MEDIUM"]
data[["Compound", "TyreLife", "LapNumber"] + features].head()

,Compound,TyreLife,LapNumber,LapNumber,Age_SOFT,Age_MEDIUM,Age_HARD,Is_SOFT,Is_MEDIUM
0,SOFT,5.0,2.0,2.0,5.0,0.0,0.0,1.0,0.0
1,SOFT,6.0,3.0,3.0,6.0,0.0,0.0,1.0,0.0
2,SOFT,7.0,4.0,4.0,7.0,0.0,0.0,1.0,0.0
3,SOFT,8.0,5.0,5.0,8.0,0.0,0.0,1.0,0.0
4,SOFT,9.0,6.0,6.0,9.0,0.0,0.0,1.0,0.0


In [50]:
race_driver = ["Year", "Round", "Driver"]
cols = ["LapTimeSec"] + features

demeaned = data[cols] - data.groupby(race_driver)[cols].transform("mean")
demeaned.head()

,LapTimeSec,LapNumber,Age_SOFT,Age_MEDIUM,Age_HARD,Is_SOFT,Is_MEDIUM
0,0.032854,-22.707317,-1.97561,-1.878049,0.0,0.268293,-0.268293
1,0.509854,-21.707317,-0.97561,-1.878049,0.0,0.268293,-0.268293
2,0.718854,-20.707317,0.02439,-1.878049,0.0,0.268293,-0.268293
3,1.029854,-19.707317,1.02439,-1.878049,0.0,0.268293,-0.268293
4,1.092854,-18.707317,2.02439,-1.878049,0.0,0.268293,-0.268293


In [51]:
X = demeaned[features].to_numpy()
y = demeaned["LapTimeSec"].to_numpy()

coefs, *_ = np.linalg.lstsq(X, y, rcond=None)

result = pd.Series(coefs, index=features).round(4)
result

LapNumber    -0.0514
Age_SOFT      0.0545
Age_MEDIUM    0.0370
Age_HARD      0.0313
Is_SOFT      -0.3576
Is_MEDIUM    -0.0336
dtype: float64

In [52]:
laps_range = np.arange(0, 31)

fuel = result["LapNumber"] * laps_range
wear = result["Age_MEDIUM"] * laps_range
net = fuel + wear

forces = pd.DataFrame({
    "Laps": laps_range,
    "Fuel burn (car gets lighter)": fuel,
    "Tyre wear (MEDIUM)": wear,
    "Net effect": net,
})

fig = px.line(
    forces, x="Laps", y=forces.columns[1:],
    labels={"value": "Lap time change (s)", "variable": ""},
    title="Two forces over a stint: fuel burn vs tyre wear (MEDIUM)",
)
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.show()

In [53]:
fig.write_image("../images/fuel_vs_wear.png", scale=2)

In [54]:
ages = np.arange(0, 31)

pace = pd.DataFrame({
    "Tyre age (laps)": ages,
    "SOFT": result["Is_SOFT"] + result["Age_SOFT"] * ages,
    "MEDIUM": result["Is_MEDIUM"] + result["Age_MEDIUM"] * ages,
    "HARD": result["Age_HARD"] * ages,
})

fig = px.line(
    pace, x="Tyre age (laps)", y=["SOFT", "MEDIUM", "HARD"],
    color_discrete_map=tyre_colors,
    labels={"value": "Slower than a fresh HARD (s)", "variable": "Tyre"},
    title="Tyre pace vs age, fuel effect removed",
)
fig.show()

In [55]:
fig.write_image("../images/soft_vs_hard_crossover.png", scale=2)